# Lab 2: Add memory to the Customer Support Agent

## Overview

Memory is a critical component of intelligence. While Large Language Models (LLMs) have impressive capabilities, they lack persistent memory across conversations. 
[Amazon Bedrock AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-getting-started.html) addresses this limitation by providing a managed service that enables AI agents to maintain context over time, remember important facts, and deliver consistent, personalized experiences.
AgentCore Memory operates on two levels:

- **Short-Term Memory**: Immediate conversation context and session-based information that provides continuity within a single interaction or closely related sessions.
- **Long-Term Memory**: Persistent information extracted and stored across multiple conversations, including facts, preferences, and summaries that enable personalized experiences over time.

In this lab, you will add memory capabilities to the Customer Support Agent implemented in Lab 1 with Amazon Bedrock AgentCore Memory. The agent will remember customer context, including order history, preferences, and previous issues, enabling more personalized and effective support. Conversations with customers are automatically stored using memory hooks, ensuring that important details are never lost.

## Architecture

![Architecture Diagram](images/architecture_lab2_memory.png)

## Prerequisites

* Python 3.12+
* AWS credentials configured  
* Anthropic Claude 4.0 enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)


## Step 1: Install Dependencies and Import Libraries

In [1]:
# Install required packages
%pip install bedrock-agentcore strands-agents boto3 -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import logging


from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

import boto3
from boto3.session import Session

boto_session = Session()
REGION = boto_session.region_name

logger = logging.getLogger(__name__)

from scripts.utils import get_ssm_parameter, put_ssm_parameter

## Step 2: Create Bedrock AgentCore Memory resources

Amazon Bedrock AgentCore Memory provides multiple long-term memory strategies. We create a memory resource combining:

- **USER_PREFERENCE**: Extracts customer preferences and behaviors
- **SEMANTIC**: Stores factual information using vector embeddings

AgentCore Memory uses namespaces to logically group long-term memory messages. Every time a new long-term memory is extracted using this memory strategy, it is saved under the namespace you set. We use the follwing namespaces using the `actorId` to group messaging of the same customer together:

- `support/customer/{actorId}/preferences`: for the user preference memory strategy
- `support/customer/{actorId}/semantic`: for the semantic memory strategy

In [3]:
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_memory_resource():
    try:
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        try:
            strategies = [
                {
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "CustomerPreferences",
                        "description": "Captures customer preferences and behavior",
                        "namespaces": ["support/customer/{actorId}/preferences"],
                    }
                },
                {
                    StrategyType.SEMANTIC.value: {
                        "name": "CustomerSupportSemantic",
                        "description": "Stores facts from conversations",
                        "namespaces": ["support/customer/{actorId}/semantic"],
                    }
                },
            ]
            # *** AGENTCORE MEMORY USAGE *** - Create memory resource with semantic strategy
            response = memory_client.create_memory_and_wait(
                name=memory_name,
                description="Customer support agent memory",
                strategies=strategies,
                event_expiry_days=90,          # Memories expire after 90 days
            )
            memory_id = response["id"]
            try:
                put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
            except:
                pass
            return memory_id
        except:
            return None


In [5]:
print("Creating AgentCore Memory resources. This can a couple of minutes..")
memory_id = create_memory_resource()
print("AgentCore Memory created successfully")

Creating AgentCore Memory resources. This can a couple of minutes..
AgentCore Memory created successfully


## Step 3: Seed previous customer 

The `create_event` action stores agent interactions into short-term memory instantly. Each saved interaction can include user messages, assistant responses, and tool actions. The process is synchronous, ensuring no conversation data is lost.

Short-term memory messages are then asynchronously processed according to the chosen long-term memory strategy.

Let's load some previously customer interactions providing the customer id as `actor_id` and a `session_id`.

In [7]:
# Seed with previous customer interactions
CUSTOMER_ID = "customer_001"

previous_interactions = [
    ("I bought a new iPhone 15 Pro. The Order number is 12345.", "USER"),
    ("Thank you for your purchase! I can see your iPhone 15 Pro order #12345 has been processed.", "ASSISTANT"),
    ("What is the warranty period for the Sennheiser headphones on June 20th. Order number 654321.", "USER"),
    ("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty.", "ASSISTANT"),
    ("I'm looking for a good laptop. I prefer ThinkPad models.", "USER"),
    ("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", "ASSISTANT")
]

# Save previous interactions
try:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Seeded customer history")
except Exception as e:
    print(f"⚠️ Error seeding history: {e}")

✅ Seeded customer history


### Visualize preferences memory

In [15]:
memories = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"support/customer/{CUSTOMER_ID}/preferences",
    query="can you summarize the support issue"
)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"  {i}. {text}")

  1. {"context":"The user explicitly stated a preference for ThinkPad laptop models during a laptop search conversation","preference":"Prefers ThinkPad laptops","categories":["technology","electronics","computers"]}


### Visualize semantic memory

In [16]:
memories = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"support/customer/{CUSTOMER_ID}/semantic",
    query="can you summarize the support issue"
)

for i, memory in enumerate(memories, 1):
    if isinstance(memory, dict):
        content = memory.get('content', {})
        if isinstance(content, dict):
            text = content.get('text', '')
            print(f"  {i}. {text}")

  1. User purchased an iPhone 15 Pro with order number 12345.
  2. User has a Sennheiser headphones order with number 654321 and a 1-year warranty.
  3. User prefers ThinkPad laptop models.


## Step 3: Implement Strands Hooks to save and retrieve agent interactions

Strands Agents provides a powerful hook system that enables components to react to or modify agent behavior through strongly-typed event callbacks. We'll use two key hook events:

- **MessageAddedEvent**: Triggered when messages are added to the conversation, allowing us to retrieve and inject customer context
- **AfterInvocationEvent**: Fired after agent responses, enabling automatic storage of interactions to memory

The hook system ensures memory operations happen automatically without manual intervention, creating a seamless experience where customer context is preserved across conversations.

To create the hooks we will extend the `HookProvider` class:


In [17]:
class CustomerSupportMemoryHooks(HookProvider):
    """Memory hooks for customer support agent"""

    def __init__(
        self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str
    ):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """Retrieve customer context before processing support query"""
        messages = event.agent.messages
        if (
            messages[-1]["role"] == "user"
            and "toolResult" not in messages[-1]["content"][0]
        ):
            user_query = messages[-1]["content"][0]["text"]

            try:
                all_context = []

                for context_type, namespace in self.namespaces.items():
                    # *** AGENTCORE MEMORY USAGE *** - Retrieve customer context from each namespace
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,
                    )
                    # Post-processing: Format memories into context strings
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(
                                        f"[{context_type.upper()}] {text}"
                                    )

                # Inject customer context into the query
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0][
                        "text"
                    ] = f"Customer Context:\n{context_text}\n\n{original_text}"
                    logger.info(f"Retrieved {len(all_context)} customer context items")

            except Exception as e:
                logger.error(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        """Save customer support interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last customer query and agent response
                customer_query = None
                agent_response = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif (
                        msg["role"] == "user"
                        and not customer_query
                        and "toolResult" not in msg["content"][0]
                    ):
                        customer_query = msg["content"][0]["text"]
                        break

                if customer_query and agent_response:
                    # *** AGENTCORE MEMORY USAGE *** - Save the support interaction
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[
                            (customer_query, "USER"),
                            (agent_response, "ASSISTANT"),
                        ],
                    )
                    logger.info("Saved support interaction to memory")

        except Exception as e:
            logger.error(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register customer support memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)
        logger.info("Customer support memory hooks registered")



## Step 4: Create a Customer Support Agent with memory

Next, we will implement the Customer Support Agent just as we did in Lab 1, but this time we instantiate the class `CustomerSupportMemoryHooks` and we pass the memory hook to the agent contructor.

In [ ]:
import uuid

from strands import Agent
from strands.models import BedrockModel

from lab_helpers.lab1_strands_agent import (
    SYSTEM_PROMPT,
    get_order_status,
    get_shipping_info,
    get_return_policy,
    get_product_info
)

SESSION_ID = str(uuid.uuid4())
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)


# Initialize the Bedrock model (Anthropic Claude 4 Sonnet)
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.3,
    region_name=REGION
)

# Create the customer support agent with all 5 tools
agent = Agent(
    model=model,
    hooks=[memory_hooks], # Pass Memory Hooks
    tools=[
        get_order_status,      # Tool 1: Simple order status lookup
        get_product_info,      # Tool 2: Simple product information lookup
        get_shipping_info,     # Tool 3: Simple shipping information lookup
        get_return_policy      # Tool 4: Simple return policy lookup
    ],
    system_prompt=SYSTEM_PROMPT
)

In [14]:
# Seed with previous customer interactions
previous_interactions = [
    ("I bought a new iPhone 15 Pro. The Order number is 12345.", "USER"),
    ("Thank you for your purchase! I can see your iPhone 15 Pro order #12345 has been processed.", "ASSISTANT"),
    ("What is the warranty period for the Sennheiser headphones on June 20th. Order number 654321.", "USER"),
    ("Perfect! I have your Sennheiser headphones order #654321 on file with the 1-year warranty.", "ASSISTANT"),
    ("I'm looking for a good laptop. I prefer ThinkPad models.", "USER"),
    ("Great choice! ThinkPads are excellent for their durability and performance. Let me help you find the right model for your needs.", "ASSISTANT")
]

# Save previous interactions
try:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Seeded customer history")
except Exception as e:
    print(f"⚠️ Error seeding history: {e}")

✅ Seeded customer history


## Step 7: Test Production Memory Hooks

Now let's test how the sophisticated MemoryHook system works automatically!

In [ ]:
response1 = agent("What is status of the shipping of my iphone?")

Let me check the shipping status for your iPhone order.
Tool #1: get_shipping_info
Here's the shipping information for your iPhone 15 Pro (Order #12345):

- **Shipping Method**: Standard Shipping (Free)
- **Estimated Delivery**: 3-5 business days
- **Current Status**: Preparing for shipment

Your iPhone is currently being prepared for shipment, which means it should be dispatched soon. Once it ships, you'll receive a tracking number to monitor its progress.

Is there anything else you'd like to know about your iPhone order or any other assistance I can provide?

In [ ]:
response1 = agent("What is my preferred Laptop?")

## Cleanup

In [ ]:
from lab_helpers.lab2_memory import delete_memory

delete_memory(memory_hooks)

## Congratulations! 🎉

You have successfully completed **Lab 2: Add memory to the Customer Support Agent**!

### What You Accomplished:

✅ Created a serverelss managed memory with Amazon Bedrock AgentCore Memory 

✅ Implemented a preference and semantic long term memory strategy

✅ Integrated AgentCore Memory with the customer support Agent using the hook mechanism provided by Strands Agents

## Resources
- [Amazon Bedrock Agent Core Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [Strands Agents Hooks Documentation](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/hooks/?h=hooks)